# Supplementary dataset audit

This notebook profiles the public supplementary collection, not official TREC training logs. It produces aggregates only; raw transcripts stay outside Git. Set `USERSIM_SUPPLEMENTARY_DIR` to the extracted `TREC UserSim` directory before execution. Requires Python and openpyxl. Source: https://next.hessenbox.de/index.php/s/TanNCrw6mZm7b6Q . Manual behavioral observations are recorded separately in the data-inspection note. Raw prompt-span counts are diagnostics, not a validated conversation parser.


In [1]:
from pathlib import Path
from collections import Counter
import os, re, statistics, hashlib
import openpyxl
root = Path(os.environ['USERSIM_SUPPLEMENTARY_DIR'])
files = sorted(root.glob('TN*_*.txt'))
assert files, 'No transcript files found'
texts = {f.stem: f.read_text() for f in files}
ids = {int(re.search(r'TN(\d+)', f.name)[1]) for f in files}
print('Transcripts:', len(files), 'Participants:', len(ids))
archive = root.parent / 'supplementary.zip'
if archive.exists():
    print('Archive SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())


Transcripts: 32 Participants: 16
Archive SHA256: 0a75fc192a853c86511756b10466794fc02c8ab5708dc603d61f9a5bb91a6590


In [2]:
book = openpyxl.load_workbook(root / 'Participant_Information.xlsx', read_only=True, data_only=True)
participants = list(book['Participants'].values)[1:]
experience = list(book['Experience'].values)[1:]
included_participants = [r for r in participants if r[0] in ids]
included_experience = [r for r in experience if r[0] in ids]
assert len({(r[0], r[1]) for r in included_experience}) == len(included_experience)
print('Workbook participant rows:', len(participants))
print('Workbook session rows:', len(experience))
print('Matched participant/session rows:', len(included_participants), len(included_experience))
print('Chatbots among included participants:', dict(Counter(r[1] for r in included_participants)))
counts = [r[2] for r in included_experience]
print('Workbook prompt count:', sum(counts))
print('Prompts per session min/median/max:', min(counts), statistics.median(counts), max(counts))


Workbook participant rows: 32
Workbook session rows: 64
Matched participant/session rows: 16 32
Chatbots among included participants: {'Perplexity': 8, 'ChatGPT Scholar GPT': 8}
Workbook prompt count: 151
Prompts per session min/median/max: 2 4.0 11


In [3]:
opening_count = sum(t.count('<PROMPT>') for t in texts.values())
strict_pairs = sum(len(re.findall(r'<PROMPT>(.*?)</PROMPT>', t, re.S)) for t in texts.values())
print('Opening PROMPT tags:', opening_count)
print('Naive matched PROMPT pairs:', strict_pairs)
tag_pattern = r'<\s*(/?)\s*(PROMPT|OUTPUT|PARTICIPANT|INTERVIEWER)\s*>'
anomalies = []
lengths = []
for name, text in texts.items():
    participant, session = map(int, re.fullmatch(r'TN(\d+)_(\d+)', name).groups())
    expected = next(r[2] for r in included_experience if r[0] == participant and r[1] == session)
    assert text.count('<PROMPT>') == expected, name
    tags = list(re.finditer(tag_pattern, text))
    for i, tag in enumerate(tags):
        if not tag[1] and tag[2] in ('PROMPT', 'OUTPUT'):
            following = tags[i + 1] if i + 1 < len(tags) else None
            if following is None or following[1] != '/' or following[2] != tag[2]:
                anomalies.append((name, text.count('\n', 0, tag.start()) + 1, tag[2]))
    spans = re.findall(r'<PROMPT>\s*(.*?)(?=<\s*/?\s*(?:PROMPT|OUTPUT|PARTICIPANT|INTERVIEWER)\s*>|\Z)', text, re.S)
    lengths.extend(len(s.split()) for s in spans)
print('All 32 per-session opening counts agree with workbook')
print('Files with unexpected boundaries:', len({a[0] for a in anomalies}))
print('Unexpected boundaries (file, line, opening tag):', anomalies)
print('Raw prompt spans:', len(lengths))
print('Whitespace word count min/median/max, annotations retained:', min(lengths), statistics.median(lengths), max(lengths))
print('Opening/closing OUTPUT tags:', sum(t.count('<OUTPUT>') for t in texts.values()), sum(t.count('</OUTPUT>') for t in texts.values()))
assert opening_count == sum(counts) == len(lengths)


Opening PROMPT tags: 151
Naive matched PROMPT pairs: 147
All 32 per-session opening counts agree with workbook
Files with unexpected boundaries: 4
Unexpected boundaries (file, line, opening tag): [('TN13_1', 314, 'OUTPUT'), ('TN22_1', 189, 'PROMPT'), ('TN22_1', 192, 'OUTPUT'), ('TN22_1', 295, 'PROMPT'), ('TN22_1', 298, 'OUTPUT'), ('TN22_1', 396, 'PROMPT'), ('TN22_1', 400, 'OUTPUT'), ('TN38_1', 107, 'PROMPT'), ('TN39_2', 5, 'OUTPUT'), ('TN39_2', 84, 'PROMPT'), ('TN39_2', 87, 'OUTPUT'), ('TN39_2', 101, 'PROMPT'), ('TN39_2', 104, 'OUTPUT')]
Raw prompt spans: 151
Whitespace word count min/median/max, annotations retained: 1 11 131
Opening/closing OUTPUT tags: 153 153


## Interpretation limits

The workbook covers participants without released transcripts. Counts of opening tags agree with the matched workbook, but balanced totals do not establish valid nesting. A naive regex can merge prompts, assistant output, and spoken commentary. The bounded spans used for word counts retain annotations and are not ready for training. Review malformed boundaries and response alternatives/regeneration before forming turn pairs. The notebook does not infer stopping labels or user personality traits.
